# OEIS Offline Matcher – Regression Notebook

This notebook runs a small, **human-auditable** regression suite for the full pipeline using `docs/regressions.json`.

It is intentionally **not** part of `pytest` because it requires a full OEIS SQLite DB (which is large).

Prereqs:
- `oeis sync`
- `oeis build-index`

By default it looks for the DB at `data/processed/oeis.db` (or `OEIS_DB_PATH`).

In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

from oeis_matcher.api import analyze_sequence

REG_PATH = Path("docs/regressions.json")
DB_PATH = Path(os.environ.get("OEIS_DB_PATH", "data/processed/oeis.db"))

if not REG_PATH.exists():
    raise FileNotFoundError(REG_PATH)
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} not found. Build it first via: oeis sync && oeis build-index, or set OEIS_DB_PATH"
    )

cases = json.loads(REG_PATH.read_text())
print(f"Loaded {len(cases)} cases from {REG_PATH}")
print(f"Using DB: {DB_PATH}")

In [ ]:
def _contains_ids(matches: list[dict], ids: list[str], *, order_matters: bool = False) -> bool:
    if not matches:
        return False
    want = list(ids)
    for m in matches:
        got = list(m.get("ids") or [])
        if order_matters:
            if got == want:
                return True
        else:
            if sorted(got) == sorted(want):
                return True
    return False


def run_case(case: dict) -> tuple[bool, float, dict]:
    name = case.get("name", "(unnamed)")
    query = case["query"]
    opts = dict(case.get("opts") or {})
    expect = dict(case.get("expect") or {})

    t0 = time.perf_counter()
    res = analyze_sequence(
        query,
        db_path=DB_PATH,
        collect_timings=True,
        **opts,
    )
    dt = time.perf_counter() - t0

    ok = True

    if "exact_top" in expect:
        top = (res["exact_matches"][0]["id"] if res.get("exact_matches") else None)
        ok = ok and (top == expect["exact_top"])

    if "transform_contains" in expect:
        ids = {m["id"] for m in (res.get("transform_matches") or [])}
        ok = ok and all(x in ids for x in expect["transform_contains"])

    if "combo_contains_ids" in expect:
        ok = ok and _contains_ids(res.get("combinations") or [], expect["combo_contains_ids"], order_matters=False)

    if "pointwise_contains_ids" in expect:
        ok = ok and _contains_ids(
            res.get("pointwise_combinations") or [],
            expect["pointwise_contains_ids"],
            order_matters=False,
        )

    if "convolution_contains_ids" in expect:
        ok = ok and _contains_ids(
            res.get("convolution_combinations") or [],
            expect["convolution_contains_ids"],
            order_matters=False,
        )

    return ok, dt, res


In [ ]:
passes = 0
fails: list[tuple[str, dict]] = []

for case in cases:
    name = case.get("name", "(unnamed)")
    ok, dt, res = run_case(case)
    status = "PASS" if ok else "FAIL"
    print(f"{status} {name} ({dt:.2f}s)")
    if ok:
        passes += 1
    else:
        fails.append((name, res))

print(f"\nSummary: {passes}/{len(cases)} passed")
if fails:
    print("Failures:")
    for name, _res in fails:
        print(f" - {name}")
